# 3. Tiền xử lý văn bản
**Làm sạch → Tokenization → POS tagging → Stop-word removal → Lemmatization**, rồi khử bài rỗng / trùng lặp.
Kết quả lưu vào `data/processed/news_clean.csv` (dùng cho notebook 4 và 5).

In [1]:
from pathlib import Path

import pandas as pd

import text_preprocessing as tp            # các hàm tiền xử lý dùng chung (xem file text_preprocessing.py)

RAW_DIR, OUT_DIR = Path("data/raw"), Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.concat([pd.read_csv(RAW_DIR / "Fake.csv").assign(label=0),    # 0 = Fake
                pd.read_csv(RAW_DIR / "True.csv").assign(label=1)],   # 1 = True
               ignore_index=True)
df[["title", "text"]] = df[["title", "text"]].fillna("")
df["raw_id"] = df.index                     # vị trí trong bảng gốc (Fake.csv nối True.csv) – để truy ngược bài gốc
print(f"{len(df):,} bài |", df["label"].map({0: "Fake", 1: "True"}).value_counts().to_dict())

44,898 bài | {'Fake': 23481, 'True': 21417}


### Các hàm tiền xử lý
Code nằm trong module [`text_preprocessing.py`](text_preprocessing.py) để **cả huấn luyện lẫn dự đoán bài mới dùng chung đúng một hàm**
(`preprocess_article(title, text)`) – nếu tiền xử lý lúc dự đoán khác lúc huấn luyện, đặc trưng sẽ lệch và mô hình dự đoán sai.

- **Làm sạch**: bỏ HTML, URL, link rút gọn (`bit.ly`, `pic.twitter.com`), tên tài khoản `@...`, ký tự không phải chữ, đưa về chữ thường.
- **Xóa "dấu vân tay" của nguồn tin**: phần EDA (notebook 2) cho thấy mỗi nguồn có câu chữ khuôn mẫu riêng, gần như chỉ xuất hiện ở một lớp.
  Nếu giữ lại, mô hình sẽ học *"bài đến từ trang nào"* thay vì *nội dung*:
  - Tin **True** (Reuters): dòng đề mục `WASHINGTON (Reuters) -`, dòng ghi công `(Reporting by ...; Editing by ...)`,
    câu tài trợ *"This article was funded in part by SAP..."*, đoạn mở đầu các bài tổng hợp tweet, tiền tố `Factbox:`.
  - Tin **Fake**: chú thích ảnh `Featured image via ...`, `Getty Images`, `screengrab`, `via YouTube/Twitter`,
    quảng cáo/chân trang `21st Century Wire`, `READ MORE ... AT:`, ghi công ảnh `Photo by .../Getty Images`,
    mã nhúng Facebook `(function(d, s, id) {...})`.
  - Link rút gọn không có `http` (`bit.ly/...` ở tin Fake, `tmsnrt.rs/...` và mã `[nL1N...]` ở tin Reuters).
- **Tokenization**: tách từ theo khoảng trắng (sau khi làm sạch chỉ còn chữ cái).
- **POS tagging**: gắn nhãn từ loại (NLTK Perceptron Tagger) **trước** khi bỏ stop-word, vì bộ gắn nhãn cần ngữ cảnh câu.
- **Stop-word removal**: bỏ stop-word tiếng Anh của NLTK và token 1 ký tự.
- **Lemmatization**: đưa từ về dạng gốc bằng WordNet **theo đúng từ loại**
  (*voted → vote*, *called → call*, *elections → election*). Nếu không truyền từ loại, WordNet coi mọi từ là danh từ
  nên động từ/tính từ giữ nguyên.

In [2]:
# Minh họa từng bước trên một bài báo
sample = df.loc[df["label"] == 1, "text"].iloc[0]
s1 = tp.clean(sample)
s2 = tp.tokenize(s1)
s3 = tp.pos_tag(s2)
s4 = tp.remove_stopwords(s3)
s5 = tp.lemmatize(s4)
print("Gốc       :", sample[:200])
print("Làm sạch  :", s1[:200])
print("Tokenize  :", s2[:15])
print("POS tag   :", s3[:15])
print("Stop-word :", [w for w, _ in s4[:15]])
print("Lemmatize :", s5[:15])

Gốc       : WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal 
Làm sạch  :  the head of a conservative republican faction in the u s congress who voted this month for a huge expansion of the national debt to pay for tax cuts called himself a fiscal conservative on sunday and
Tokenize  : ['the', 'head', 'of', 'a', 'conservative', 'republican', 'faction', 'in', 'the', 'u', 's', 'congress', 'who', 'voted', 'this']
POS tag   : [('the', 'DT'), ('head', 'NN'), ('of', 'IN'), ('a', 'DT'), ('conservative', 'JJ'), ('republican', 'JJ'), ('faction', 'NN'), ('in', 'IN'), ('the', 'DT'), ('u', 'JJ'), ('s', 'JJ'), ('congress', 'NN'), ('who', 'WP'), ('voted', 'VBD'), ('this', 'DT')]
Stop-word : ['head', 'conservative', 'republican', 'faction', 'congress', 'voted', 'month', 'huge', 'expansion', 'national', 'debt', 'pay', 'tax', 'cuts', 'called']
Lemmatize 

In [3]:
# Áp dụng cho toàn bộ dữ liệu, chạy song song trên mọi nhân CPU (POS tagging khá chậm: ~2-3 phút).
# Tiêu đề và thân bài xử lý riêng (giống hệt preprocess_article) và giữ riêng thân bài để khử trùng theo thân bài.
df["title_clean"] = tp.parallel_preprocess(df["title"])
df["body_clean"] = tp.parallel_preprocess(df["text"])
df["clean_text"] = (df["title_clean"] + " " + df["body_clean"]).str.strip()
df["title_only"] = df["body_clean"].eq("")          # thân bài rỗng (hoặc rỗng sau khi làm sạch) – chỉ còn tiêu đề

### Khử bài rỗng và trùng lặp
Thứ tự các bước (bảng bên dưới cho biết mỗi bước loại bao nhiêu bài):
1. Bỏ bài rỗng cả tiêu đề lẫn thân bài.
2. Bỏ các bài **cùng nội dung nhưng khác nhãn** – xét cả theo *thân bài* và theo *tiêu đề + thân bài* (nhãn mâu thuẫn, không biết bài nào đúng).
3. **Khử trùng theo thân bài**: cùng thân bài nhưng khác tiêu đề vẫn là một bài (nhiều trang đăng lại một bài với tiêu đề khác).
   Nếu chỉ khử trùng theo chuỗi *tiêu đề + thân bài*, hai bản này có thể rơi một bên vào train, một bên vào test → rò rỉ.
4. Khử trùng theo *tiêu đề + thân bài* cho các bài chỉ có tiêu đề.

Bài **chỉ có tiêu đề** (thân bài rỗng) **được giữ lại** vì tiêu đề vẫn mang nội dung, và được đánh dấu ở cột `title_only`
để có thể lọc riêng khi cần. Kiểm tra bài **gần trùng** (khác vài từ) giữa train và test nằm ở notebook 4.

In [4]:
steps = [("Ban đầu", len(df))]

df = df[df["clean_text"] != ""]
steps.append(("1. Bỏ bài rỗng cả tiêu đề lẫn thân bài", len(df)))

conflict_body = ~df["title_only"] & (df.groupby("body_clean")["label"].transform("nunique") > 1)
conflict_full = df.groupby("clean_text")["label"].transform("nunique") > 1
df = df[~(conflict_body | conflict_full)]
steps.append(("2. Bỏ bài cùng nội dung nhưng khác nhãn", len(df)))

df = df[df["title_only"] | ~df.duplicated("body_clean")]
steps.append(("3. Khử trùng theo thân bài", len(df)))

df = df.drop_duplicates("clean_text")
steps.append(("4. Khử trùng theo tiêu đề + thân bài", len(df)))

steps = pd.DataFrame(steps, columns=["Bước", "Số bài còn lại"])
steps["Số bài bị loại"] = (-steps["Số bài còn lại"].diff()).fillna(0).astype(int)
display(steps)

label_name = df["label"].map({0: "Fake", 1: "True"})
print(f"Còn {len(df):,} bài |", label_name.value_counts().to_dict())
print(f"Trong đó {df['title_only'].sum():,} bài chỉ có tiêu đề (được giữ lại):",
      label_name[df["title_only"]].value_counts().to_dict())

df[["raw_id", "title", "subject", "label", "title_only", "clean_text"]].to_csv(OUT_DIR / "news_clean.csv", index=False)
df[["title", "clean_text"]].head()

,Bước,Số bài còn lại,Số bài bị loại
0,Ban đầu,44898,0
1,1. Bỏ bài rỗng cả tiêu đề lẫn thân bài,44889,9
2,2. Bỏ bài cùng nội dung nhưng khác nhãn,44889,0
3,3. Khử trùng theo thân bài,39032,5857
4,4. Khử trùng theo tiêu đề + thân bài,38821,211


Còn 38,821 bài | {'True': 20935, 'Fake': 17886}
Trong đó 496 bài chỉ có tiêu đề (được giữ lại): {'Fake': 495, 'True': 1}


,title,clean_text
0,Donald Trump Sends Out Embarrassing New Year’...,donald trump send embarrass new year eve messa...
1,Drunk Bragging Trump Staffer Started Russian ...,drunk brag trump staffer start russian collusi...
2,Sheriff David Clarke Becomes An Internet Joke...,sheriff david clarke become internet joke thre...
3,Trump Is So Obsessed He Even Has Obama’s Name...,trump obsessed even obama name cod website ima...
4,Pope Francis Just Called Out Donald Trump Dur...,pope francis call donald trump christmas speec...


### Kiểm tra: dấu hiệu nguồn tin đã bị loại chưa?
Tỷ lệ % bài (theo lớp) còn chứa các cụm từ khuôn mẫu **sau** tiền xử lý. Lý tưởng là ≈ 0 ở cả hai lớp.

In [5]:
MARKERS = ["reuters", "featured image", "getty image", "image via", "screengrab", "century wire",
           "factbox", "independently created", "source link", "somodevilla", "fjs", "tmsnrt", "filessupport", "cdata", "subscribe become"]
pd.DataFrame({m: df["clean_text"].str.contains(rf"\b{m}\b").groupby(label_name).mean() * 100
              for m in MARKERS}).round(2)

,reuters,featured image,getty image,image via,screengrab,century wire,factbox,independently created,source link,somodevilla,fjs,tmsnrt,filessupport,cdata,subscribe become
label,,,,,,,,,,,,,,,
Fake,0.01,0.01,0.01,0.03,0.0,0.02,0.00,0.0,0.01,0.52,0.04,0.0,0.0,0.0,0.81
True,0.00,0.00,0.00,0.00,0.0,0.00,0.09,0.0,0.00,0.00,0.00,0.0,0.0,0.0,0.00
